# ECG Aritmisi Sınıflandırıcı — Kaggle GPU Eğitimi
**Proje:** TÜBİTAK 2209-A — Hayatın Ritmi  
**Dataset:** PhysioNet SPH 12-Lead ECG Arrhythmia (45,152 kayıt)  
**Model:** Depthwise Separable 1D CNN → TFLite INT8 → Android  

## Kurulum
1. Bu notebook'u Kaggle'a yükle
2. **Settings → Secrets** → `HF_TOKEN` ekle (HuggingFace Write token)
3. **GPU T4 x2** seç → **Run All**
4. Model otomatik olarak `adzetto/ecg-arrhythmia-classifier`'a push edilir

In [ ]:
# Bağımlılıkları kur
!pip install wfdb huggingface_hub -q
print('Kurulum tamamlandı')

In [ ]:
import os, csv, time
import numpy as np
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi

# Kaggle secret'tan token oku
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
HF_REPO_ID = 'adzetto/ecg-arrhythmia-classifier'

# GPU kontrolü
gpus = tf.config.list_physical_devices('GPU')
print('TF version:', tf.__version__)
print('GPU listesi:', gpus)
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print('✅ GPU aktif:', gpus[0].name)

In [ ]:
# PhysioNet ECG Arrhythmia dataset'ini indir (wfdb üzerinden)
DATASET_PATH = '/kaggle/working/ecg-arrhythmia'

if not os.path.exists(DATASET_PATH):
    print('Dataset indiriliyor...')
    import wfdb
    wfdb.dl_database('ecg-arrhythmia', dl_dir=DATASET_PATH)
    print('✅ Dataset indirildi')
else:
    print('✅ Dataset zaten mevcut')

SNOMED_CSV   = os.path.join(DATASET_PATH, 'ConditionNames_SNOMED-CT.csv')
RECORDS_FILE = os.path.join(DATASET_PATH, 'RECORDS')
print('Kayıt dosyası:', RECORDS_FILE)

In [ ]:
# ─── Konfigürasyon ───────────────────────────────────────────
FS_ORIGINAL = 500
FS_TARGET   = 250
N_SAMPLES   = FS_TARGET * 10  # 2500
N_LEADS     = 12
BATCH_SIZE  = 128   # GPU'da daha büyük batch
EPOCHS      = 50
MODEL_DIR   = '/kaggle/working/models'
os.makedirs(MODEL_DIR, exist_ok=True)

# SNOMED-CT label map
def load_snomed_map():
    codes, names = [], []
    with open(SNOMED_CSV, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            codes.append(int(row['Snomed_CT']))
            names.append(row['Acronym Name'])
    return {code: i for i, code in enumerate(codes)}, names

SNOMED_MAP, CONDITION_NAMES = load_snomed_map()
NUM_CLASSES = len(SNOMED_MAP)
print(f'✅ {NUM_CLASSES} kondisyon, {N_SAMPLES} sample/kayıt, batch={BATCH_SIZE}')

In [ ]:
from scipy.signal import butter, filtfilt
import wfdb

def bandpass_filter(sig, fs=500, lo=0.5, hi=40.0, order=2):
    nyq = fs / 2.0
    b, a = butter(order, [lo/nyq, hi/nyq], btype='band')
    return filtfilt(b, a, sig, axis=-1)

def preprocess(sig_12xN):
    sig = bandpass_filter(sig_12xN)
    sig = sig[:, ::2]                                          # 500→250Hz
    sig = (sig - sig.mean(1, keepdims=True)) / (sig.std(1, keepdims=True) + 1e-8)
    return sig.T.astype(np.float32)                            # (2500, 12)

def parse_dx(hea_path):
    codes = []
    try:
        with open(hea_path, encoding='utf-8') as f:
            for line in f:
                if line.startswith('#Dx:'):
                    for c in line.split(':')[1].strip().split(','):
                        try: codes.append(int(c.strip()))
                        except ValueError: pass
    except Exception: pass
    return codes

def load_record(rec_path):
    try:
        record = wfdb.rdrecord(rec_path)
        if record.p_signal.shape != (5000, 12):
            return None, None
        x = preprocess(record.p_signal.T)
        label = np.zeros(NUM_CLASSES, dtype=np.float32)
        for code in parse_dx(rec_path + '.hea'):
            if code in SNOMED_MAP:
                label[SNOMED_MAP[code]] = 1.0
        return x, label
    except Exception:
        return None, None

print('✅ Ön işleme fonksiyonları hazır')

In [ ]:
print('📂 Dataset yükleniyor (45K kayıt)...')
t0 = time.time()

with open(RECORDS_FILE, encoding='utf-8') as f:
    folders = [l.strip() for l in f if l.strip()]

X_list, Y_list = [], []
skipped = 0

for i, folder in enumerate(folders):
    folder_path = os.path.join(DATASET_PATH, folder)
    if not os.path.isdir(folder_path):
        continue
    for fname in sorted(os.listdir(folder_path)):
        if fname.endswith('.hea'):
            x, y = load_record(os.path.join(folder_path, fname[:-4]))
            if x is not None:
                X_list.append(x)
                Y_list.append(y)
            else:
                skipped += 1
    if i % 5 == 0:
        elapsed = time.time() - t0
        print(f'  [{i+1}/{len(folders)}] {len(X_list)} kayıt, {elapsed:.0f}s', end='\r')

X = np.array(X_list, dtype=np.float32)  # (N, 2500, 12)
Y = np.array(Y_list, dtype=np.float32)  # (N, 49)
print(f'\n✅ X:{X.shape}  Y:{Y.shape}  atlandı:{skipped}  süre:{time.time()-t0:.0f}s')
print(f'   Pozitif label oranı: {Y.mean():.4f}')

In [ ]:
def ds_conv_block(x, filters, kernel_size=7, strides=1):
    x = layers.DepthwiseConv1D(kernel_size, strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(6.0)(x)
    x = layers.Conv1D(filters, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(6.0)(x)
    return x

def build_ecg_model(input_shape=(N_SAMPLES, N_LEADS), num_classes=NUM_CLASSES):
    inp = layers.Input(shape=input_shape, name='ecg_input')
    x = layers.Conv1D(32, 15, strides=2, padding='same', use_bias=False)(inp)  # →(1250,32)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(6.0)(x)
    x = ds_conv_block(x, 64,  kernel_size=7, strides=2)  # →(625, 64)
    x = ds_conv_block(x, 128, kernel_size=7, strides=2)  # →(313,128)
    x = ds_conv_block(x, 128, kernel_size=5, strides=2)  # →(157,128)
    x = ds_conv_block(x, 256, kernel_size=5, strides=2)  # →( 79,256)
    x = ds_conv_block(x, 256, kernel_size=3, strides=2)  # →( 40,256)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(6.0)(x)
    out = layers.Dense(num_classes, activation='sigmoid', name='predictions')(x)
    return Model(inp, out, name='EcgDSCNN_v1')

model = build_ecg_model()
model.summary()
print(f'✅ Parametre sayısı: {model.count_params():,}')

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.15, random_state=42)
print(f'Train: {len(X_train)}, Val: {len(X_val)}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=[
        tf.keras.metrics.AUC(multi_label=True, num_labels=NUM_CLASSES, name='auc'),
        tf.keras.metrics.BinaryAccuracy(name='acc', threshold=0.5),
    ]
)

cb_list = [
    callbacks.ReduceLROnPlateau(monitor='val_auc', patience=4, factor=0.5, mode='max', verbose=1),
    callbacks.EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(f'{MODEL_DIR}/ecg_best.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    callbacks.CSVLogger(f'{MODEL_DIR}/training_log.csv'),
]

print('🚂 Eğitim başlıyor...')
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cb_list,
)

best_auc = max(history.history['val_auc'])
print(f'\n🏆 En iyi val AUC: {best_auc:.4f}')

In [ ]:
print('📦 TFLite INT8 export...')

def representative_dataset():
    idx = np.random.choice(len(X_val), 300, replace=False)
    for i in idx:
        yield [X_val[i:i+1]]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()
tflite_path = f'{MODEL_DIR}/ecg_model_int8.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f'✅ TFLite boyutu: {len(tflite_model)/1024:.1f} KB')

In [ ]:
print(f'🤗 HuggingFace\'e yükleniyor: {HF_REPO_ID}')
api = HfApi(token=HF_TOKEN)

# TFLite model
api.upload_file(
    path_or_fileobj=tflite_path,
    path_in_repo='ecg_model_int8.tflite',
    repo_id=HF_REPO_ID, repo_type='model', token=HF_TOKEN,
    commit_message=f'Upload INT8 TFLite model (val_auc={best_auc:.4f})'
)

# Keras model
api.upload_file(
    path_or_fileobj=f'{MODEL_DIR}/ecg_best.keras',
    path_in_repo='ecg_best.keras',
    repo_id=HF_REPO_ID, repo_type='model', token=HF_TOKEN,
    commit_message='Upload Keras model'
)

# Training log
api.upload_file(
    path_or_fileobj=f'{MODEL_DIR}/training_log.csv',
    path_in_repo='training_log.csv',
    repo_id=HF_REPO_ID, repo_type='model', token=HF_TOKEN,
    commit_message='Upload training log'
)

# SNOMED label dosyası (Android için)
import json
label_map = {
    'conditions': CONDITION_NAMES,
    'num_classes': NUM_CLASSES,
    'input_shape': [N_SAMPLES, N_LEADS],
    'sample_rate_hz': FS_TARGET,
    'best_val_auc': best_auc,
}
label_json = json.dumps(label_map, indent=2, ensure_ascii=False)
api.upload_file(
    path_or_fileobj=label_json.encode(),
    path_in_repo='model_config.json',
    repo_id=HF_REPO_ID, repo_type='model', token=HF_TOKEN,
    commit_message='Upload model config with label names'
)

print(f'\n✅ TAMAMLANDI!')
print(f'   Model : https://huggingface.co/{HF_REPO_ID}')
print(f'   Val AUC: {best_auc:.4f}')